In [ ]:
# Imports

device="cuda"

from pytorch3d.renderer import (
    FoVPerspectiveCameras, look_at_view_transform,
    RasterizationSettings, BlendParams,
    MeshRenderer, MeshRasterizer, HardPhongShader, PointLights
)


lights = PointLights(
    device=device,
    location=[[2.0, 2.0, -2.0]]  # position of the light
)

# Initialize an OpenGL perspective camera.
R, T = look_at_view_transform(2.7, 10, 20)
cameras = FoVPerspectiveCameras(device=device, R=R, T=T)

# Define the settings for rasterization and shading. Here we set the output image to be of size
# 512x512. As we are rendering images for visualization purposes only we will set faces_per_pixel=1
# and blur_radius=0.0. Refer to rasterize_meshes.py for explanations of these parameters.
raster_settings = RasterizationSettings(
    image_size=512,
    blur_radius=0.0,
    faces_per_pixel=1,
)

# Create a Phong renderer by composing a rasterizer and a shader. Here we can use a predefined
# PhongShader, passing in the device on which to initialize the default parameters
renderer = MeshRenderer(
    rasterizer=MeshRasterizer(
        cameras=cameras,
        raster_settings=raster_settings
    ),
    shader=HardPhongShader(
        device=device,
        cameras=cameras,
        lights=lights
    )
)

In [ ]:
import torch
import numpy as np
from pytorch3d.utils import ico_sphere
from pytorch3d.renderer import TexturesVertex

mesh = ico_sphere(level=3, device=device)

# White color for every vertex
verts_rgb = torch.ones_like(mesh.verts_padded(), device=device)

mesh.textures = TexturesVertex(verts_features=verts_rgb)

images = renderer(mesh)

import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
plt.imshow(images[0, ..., :3].detach().cpu())
plt.axis("off")
plt.show()

In [ ]:
import torch
import torch.nn as nn

from pytorch3d.renderer import BlendParams, hard_rgb_blend
from pytorch3d.ops import interpolate_face_attributes


class HardSHShader(nn.Module):
    def __init__(self, device="cuda", cameras=None, blend_params=None, sh_coeffs=None):
        super().__init__()
        self.device = device
        self.cameras = cameras
        self.blend_params = blend_params or BlendParams()

        if sh_coeffs is None:
            # 9 SH coefficients, RGB.
            # Mostly ambient + a bit of directional-looking light.
            sh_coeffs = torch.tensor(
                [
                    [0.8, 0.8, 0.8],   # l=0
                    [0.0, 0.0, 0.0],   # y
                    [0.3, 0.3, 0.3],   # z
                    [0.2, 0.2, 0.2],   # x
                    [0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0],
                ],
                device=device,
            )

        self.register_buffer("sh_coeffs", sh_coeffs)

    def sh_basis(self, normals):
        x = normals[..., 0]
        y = normals[..., 1]
        z = normals[..., 2]

        return torch.stack(
            [
                0.282095 * torch.ones_like(x),
                0.488603 * y,
                0.488603 * z,
                0.488603 * x,
                1.092548 * x * y,
                1.092548 * y * z,
                0.315392 * (3.0 * z * z - 1.0),
                1.092548 * x * z,
                0.546274 * (x * x - y * y),
            ],
            dim=-1,
        )

    def forward(self, fragments, meshes, **kwargs):
        faces = meshes.faces_packed()
        vertex_normals = meshes.verts_normals_packed()
        face_normals = vertex_normals[faces]

        pixel_normals = interpolate_face_attributes(
            fragments.pix_to_face,
            fragments.bary_coords,
            face_normals,
        )

        pixel_normals = torch.nn.functional.normalize(pixel_normals, dim=-1)

        # [N, H, W, K, 9]
        basis = self.sh_basis(pixel_normals)

        # [N, H, W, K, 3]
        lighting = torch.einsum("...b,bc->...c", basis, self.sh_coeffs)
        lighting = torch.clamp(lighting, min=0.0)

        # Uses mesh vertex/texture color as albedo
        albedo = meshes.sample_textures(fragments)

        colors = albedo * lighting

        return hard_rgb_blend(colors, fragments, self.blend_params)

In [ ]:
renderer = MeshRenderer(
    rasterizer=MeshRasterizer(
        cameras=cameras,
        raster_settings=raster_settings,
    ),
    shader=HardSHShader(
        device=device,
        cameras=cameras,
    ),
)

In [ ]:
images = renderer(mesh)

plt.figure(figsize=(6, 6))
plt.imshow(images[0, ..., :3].detach().cpu().numpy())
plt.axis("off")
plt.show()

In [ ]:
import torch
import torch.nn as nn
from pytorch3d.ops import interpolate_face_attributes
from pytorch3d.renderer import BlendParams, hard_rgb_blend


class HardNormalShader(nn.Module):
    def __init__(self, device="cuda", blend_params=None):
        super().__init__()
        self.device = device
        self.blend_params = blend_params or BlendParams()

    def forward(self, fragments, meshes, **kwargs):
        faces = meshes.faces_packed()
        verts_normals = meshes.verts_normals_packed()
        faces_normals = verts_normals[faces]

        pixel_normals = interpolate_face_attributes(
            fragments.pix_to_face,
            fragments.bary_coords,
            faces_normals,
        )

        pixel_normals = torch.nn.functional.normalize(pixel_normals, dim=-1)

        # Convert normals from [-1, 1] to [0, 1]
        colors = 0.5 * pixel_normals + 0.5

        return hard_rgb_blend(colors, fragments, self.blend_params)

In [ ]:
normal_renderer = MeshRenderer(
    rasterizer=MeshRasterizer(
        cameras=cameras,
        raster_settings=raster_settings,
    ),
    shader=HardNormalShader(device=device),
)

normal_image = normal_renderer(mesh)

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(normal_image[0, ..., :3].detach().cpu().numpy())
plt.axis("off")
plt.show()

In [ ]:
import torch.nn as nn
from pytorch3d.renderer import BlendParams, hard_rgb_blend


class HardAlbedoShader(nn.Module):
    def __init__(self, blend_params=None):
        super().__init__()
        self.blend_params = blend_params or BlendParams()

    def forward(self, fragments, meshes, **kwargs):
        albedo = meshes.sample_textures(fragments)
        return hard_rgb_blend(albedo, fragments, self.blend_params)

In [ ]:
albedo_renderer = MeshRenderer(
    rasterizer=MeshRasterizer(
        cameras=cameras,
        raster_settings=raster_settings,
    ),
    shader=HardAlbedoShader(),
)

albedo_image = albedo_renderer(mesh)

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(albedo_image[0, ..., :3].detach().cpu().numpy())
plt.axis("off")
plt.show()